# SVN-FCA-IC (second version): nine binary datasets

**How to run:** Colab menu *Runtime, then Run all*. Internet access is needed (Colab has it). Expected run time: about 25 to 45 minutes.

**Datasets and their roles.** *Development* datasets (Pima, Haberman, Bupa, Appendicitis, Ionosphere) were looked at while the second version of the rule filter was designed. *Confirmation* datasets (Statlog heart, blood transfusion, Indian liver) were not used to design or tune the second version. WDBC (first five mean features) is the reference from the main notebook. The roles are reported in the paper.

**Protocol.** Repeated stratified 5-fold cross-validation, 10 repeats. For datasets with more than five features, the five features with the largest ANOVA F value are chosen inside each training fold. The positive (M) class is the disease or minority class; the printed class counts show which original label was used. Rows with missing values are removed and non-numeric columns are dropped; every such step is printed.

**What to send back:** `svnfca_v2_extra_datasets_results.zip`. If a dataset fails to download, the reason is printed and the others continue.

In [ ]:
%pip -q install wittgenstein
import os, sys, json, time, platform, warnings
import numpy as np, pandas as pd, scipy, sklearn, matplotlib
warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)
env = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
           scikit_learn=sklearn.__version__, matplotlib=matplotlib.__version__)
try:
    import wittgenstein; env["wittgenstein"] = getattr(wittgenstein, "__version__", "installed")
except Exception as e:
    env["wittgenstein"] = str(e)
json.dump(env, open("results/environment.json", "w"), indent=1)
print(env)

## Model and evaluation code (written to three small files)

In [ ]:
%%writefile svnfca.py
"""Clean re-implementation of the SVN-FCA-IC pipeline found in the ChatGPT notebook.
All preprocessing parameters are learned on training data only."""
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer

CATS = ["low", "medium", "high"]


def load():
    d = load_breast_cancer()
    X = pd.DataFrame(d.data, columns=d.feature_names)
    y = pd.Series(np.where(d.target == 0, "M", "B"), name="diagnosis")
    return X, y


# ---------- crisp (tertile) context ----------
def fit_tertile_edges(Xtr):
    return np.quantile(Xtr, [1 / 3, 2 / 3], axis=0).T  # (p,2)


def crisp_context(X, edges):
    n, p = X.shape
    C = np.zeros((n, 3 * p), bool)
    for j in range(p):
        q1, q2 = edges[j]
        cat = np.where(X[:, j] <= q1, 0, np.where(X[:, j] <= q2, 1, 2))
        C[np.arange(n), 3 * j + cat] = True
    return C


# ---------- neutrosophic (SVNS) layer ----------
def fit_knots(Xtr):
    return np.quantile(Xtr, [0.2, 0.5, 0.8], axis=0).T  # (p,3)


def memberships(X, knots):
    n, p = X.shape
    T = np.zeros((n, p, 3))
    for j in range(p):
        a, b, c = knots[j]
        x = X[:, j]
        T[:, j, 0] = np.clip((b - x) / (b - a), 0, 1)
        left = np.clip((x - a) / (b - a), 0, 1)
        right = np.clip((c - x) / (c - b), 0, 1)
        T[:, j, 1] = np.minimum(left, right)
        T[:, j, 2] = np.clip((x - b) / (c - b), 0, 1)
    return T


def indeterminacy(T, kind="max"):
    if kind == "max":
        return 1.0 - T.max(axis=2)
    if kind == "entropy":
        with np.errstate(divide="ignore", invalid="ignore"):
            h = -np.where(T > 0, T * np.log(T), 0.0).sum(axis=2)
        return h / np.log(3)
    if kind == "none":
        return np.zeros(T.shape[:2])
    raise ValueError(kind)


def evidence_matrix(X, knots, ikind="max"):
    T = memberships(X, knots)
    I = indeterminacy(T, ikind)
    E = T * (1.0 - I[:, :, None])
    n, p, _ = E.shape
    return E.reshape(n, 3 * p), T, I


# ---------- rule mining (Apriori-style, identical rule set to brute force) ----------
def mine_rules(C, y, min_extent=10, min_purity=0.85, max_len=5, closed_only=False):
    n, m = C.shape
    yM = (np.asarray(y) == "M")
    feat = np.arange(m) // 3
    out = []
    level = [((j,), C[:, j]) for j in range(m) if C[:, j].sum() >= min_extent]
    k = 1
    while level and k <= max_len:
        for items, ext in level:
            size = int(ext.sum())
            nM = int((ext & yM).sum())
            nB = size - nM
            maj = "M" if nM > nB else "B"
            pur = max(nM, nB) / size
            if pur >= min_purity:
                closed = int(C[ext].all(axis=0).sum()) == len(items)
                if closed or not closed_only:
                    out.append(dict(items=items, cls=maj, extent=size,
                                    support=size / n, purity=pur, closed=closed,
                                    ext=ext.copy()))
        if k == max_len:
            break
        nxt = []
        for items, ext in level:
            last = items[-1]
            for j in range(last + 1, m):
                if feat[j] <= feat[last]:
                    continue
                e2 = ext & C[:, j]
                if e2.sum() >= min_extent:
                    nxt.append((items + (j,), e2))
        level = nxt
        k += 1
    return out


class RuleBase:
    def __init__(self, rules, m):
        self.rules = rules
        self.m = m
        r = len(rules)
        self.R = np.zeros((r, m))
        for i, ru in enumerate(rules):
            self.R[i, list(ru["items"])] = 1
        self.len = self.R.sum(1)
        self.isM = np.array([ru["cls"] == "M" for ru in rules], bool)
        self.pur = np.array([ru["purity"] for ru in rules])

    def __len__(self):
        return len(self.rules)

    def soft_scores(self, E):
        """geometric mean of per-condition evidence; 0 if any condition has no evidence."""
        pos = E > 1e-12
        LE = np.where(pos, np.log(np.where(pos, E, 1.0)), 0.0)
        Z = (~pos).astype(float)
        nz = Z @ self.R.T
        sl = LE @ self.R.T
        return np.where(nz == 0, np.exp(sl / self.len), 0.0)

    def crisp_scores(self, C):
        return ((C.astype(float) @ self.R.T) == self.len).astype(float)

    def aggregate(self, S):
        W = S * self.pur
        b = W[:, ~self.isM].sum(1)
        mm = W[:, self.isM].sum(1)
        return b, mm


def decide(b, m):
    tot = b + m
    pred = np.where(b > m, "B", "M").astype(object)
    pred[(tot == 0) | np.isclose(b, m)] = "abstain"
    margin = np.where(tot > 0, (b - m) / np.where(tot > 0, tot, 1), 0.0)
    return pred, margin


def run_svn(rb, Xte, Cte, knots, mode="svn", ikind="max"):
    if mode == "crisp":
        S = rb.crisp_scores(Cte)
    else:
        E, _, _ = evidence_matrix(Xte, knots, "none" if mode == "T" else ikind)
        S = rb.soft_scores(E)
    b, m = rb.aggregate(S)
    pred, margin = decide(b, m)
    return pred, margin, b, m


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred, dtype=object)
    acc_mask = y_pred != "abstain"
    cov = acc_mask.mean()
    yt, yp = y_true[acc_mask], y_pred[acc_mask]
    if len(yt) == 0:
        return dict(coverage=cov)
    tp = int(((yt == "M") & (yp == "M")).sum())
    fn = int(((yt == "M") & (yp == "B")).sum())
    fp = int(((yt == "B") & (yp == "M")).sum())
    tn = int(((yt == "B") & (yp == "B")).sum())
    rec = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(coverage=cov, accuracy=(tp + tn) / len(yt), balanced_accuracy=(rec + spec) / 2,
                precision_M=prec, recall_M=rec, specificity=spec, f1_M=f1,
                tp=tp, fn=fn, fp=fp, tn=tn, n_accepted=len(yt))


In [ ]:
%%writefile svnfca2.py
"""SVN-FCA-IC, version 2: class-prior-aware rule filter, top-k class evidence and
tertile-consistent neutrosophic memberships. Every learned quantity (cut points, knots, rules,
filter parameters) is estimated from the training data only; filter parameters are chosen by
inner cross-validation on the training data."""
import math
import numpy as np
from sklearn.model_selection import StratifiedKFold
from svnfca import crisp_context, fit_tertile_edges

EPS = 1e-12


# ---------- neutrosophic memberships consistent with the tertile context ----------
def fit_knots_consistent(Xtr):
    """Knots (a, b, c) such that the crossing points of the memberships are the tertile cut points
    and the peak of 'medium' is the median: (a+b)/2 = q33, b = q50, (b+c)/2 = q67."""
    q = np.quantile(Xtr, [1 / 3, .5, 2 / 3], axis=0).T
    return np.stack([2 * q[:, 0] - q[:, 1], q[:, 1], 2 * q[:, 2] - q[:, 1]], axis=1)


def fit_knots_percentile(Xtr):
    return np.quantile(Xtr, [.2, .5, .8], axis=0).T


def memberships(X, knots):
    n, p = X.shape
    T = np.zeros((n, p, 3))
    for j in range(p):
        a, b, c = knots[j]
        wl, wr = max(b - a, EPS), max(c - b, EPS)
        x = X[:, j]
        T[:, j, 0] = np.clip((b - x) / wl, 0, 1)
        T[:, j, 1] = np.minimum(np.clip((x - a) / wl, 0, 1), np.clip((c - x) / wr, 0, 1))
        T[:, j, 2] = np.clip((x - b) / wr, 0, 1)
    return T


def evidence(X, knots, kind="max"):
    """kind: 'max' (I = 1 - max T), 'none' (I = 0, truth only) or 'entropy' (normalised entropy of the memberships)."""
    T = memberships(X, knots)
    if kind == "max":
        I = 1.0 - T.max(axis=2)
    elif kind == "entropy":
        with np.errstate(divide="ignore", invalid="ignore"):
            I = -np.where(T > 0, T * np.log(T), 0.0).sum(axis=2) / np.log(3)
    else:
        I = np.zeros(T.shape[:2])
    E = T * (1.0 - I[:, :, None])
    return E.reshape(len(X), -1)


# ---------- candidate antecedents ----------
def mine_candidates(C, min_extent, max_len=5):
    """All attribute sets with at most one category per feature, at least min_extent objects, length <= max_len."""
    n, m = C.shape
    feat = np.arange(m) // 3
    out, level, k = [], [((j,), C[:, j]) for j in range(m) if C[:, j].sum() >= min_extent], 1
    while level and k <= max_len:
        out += level
        if k == max_len:
            break
        nxt = []
        for items, ext in level:
            last = items[-1]
            for j in range(last + 1, m):
                if feat[j] <= feat[last]:
                    continue
                e2 = ext & C[:, j]
                if e2.sum() >= min_extent:
                    nxt.append((items + (j,), e2))
        level, k = nxt, k + 1
    return out


class Candidates:
    def __init__(self, C, yM, min_extent, max_len=5):
        cand = mine_candidates(C, min_extent, max_len)
        self.m = C.shape[1]
        self.items = [c[0] for c in cand]
        self.R = np.zeros((len(cand), self.m))
        for i, it in enumerate(self.items):
            self.R[i, list(it)] = 1
        self.len = self.R.sum(1)
        ext = np.array([c[1] for c in cand]) if cand else np.zeros((0, len(C)), bool)
        self.size = ext.sum(1).astype(float)
        self.nM = (ext & yM[None, :]).sum(1).astype(float)
        self.piM = float(yM.mean())
        self._C, self._ext, self._closed = C, ext, None

    @property
    def closed(self):
        if self._closed is None:
            self._closed = np.array([int(self._C[e].all(axis=0).sum()) == len(it) for it, e in zip(self.items, self._ext)], bool) if len(self.items) else np.zeros(0, bool)
        return self._closed

    def select(self, delta, relative=True):
        """Return (mask, isM, weight). Laplace-smoothed class purity; relative = excess over the class prior."""
        pM = (self.nM + 1) / (self.size + 2)
        pB = 1 - pM
        if relative:
            rM = (pM - self.piM) / max(1 - self.piM, EPS)
            rB = (pB - (1 - self.piM)) / max(self.piM, EPS)
        else:
            rM, rB = pM, pB
        isM = rM >= rB
        q = np.where(isM, rM, rB)
        return q >= delta, isM, q

    def soft(self, E):
        pos = E > 1e-12
        LE = np.where(pos, np.log(np.where(pos, E, 1.0)), 0.0)
        nz = (~pos).astype(float) @ self.R.T
        return np.where(nz == 0, np.exp(LE @ self.R.T / self.len), 0.0)

    def crisp(self, C):
        return ((C.astype(float) @ self.R.T) == self.len).astype(float)


def aggregate(S, mask, isM, w, k, agg):
    """Class evidence. 'topk': mean of the k strongest weighted rule scores; 'sum': sum of all."""
    W = S * w[None, :]
    out, used = [], []
    for cls_mask in (mask & ~isM, mask & isM):
        Wc = W[:, cls_mask]
        if Wc.shape[1] == 0:
            out.append(np.zeros(len(S))); used.append(np.zeros(len(S)))
        elif agg == "sum":
            out.append(Wc.sum(1)); used.append((Wc > 0).sum(1))
        else:
            kk = min(k, Wc.shape[1])
            top = -np.sort(-Wc, axis=1)[:, :kk]
            out.append(top.sum(1) / k); used.append(np.minimum((Wc > 0).sum(1), k))
    return out[0], out[1], used[0], used[1]  # evidence B, evidence M, rules used B, rules used M


def decide(b, m):
    tot = b + m
    pred = np.where(b > m, "B", "M").astype(object)
    pred[(tot == 0) | np.isclose(b, m)] = "abstain"
    margin = np.where(tot > 0, (b - m) / np.where(tot > 0, tot, 1), 0.0)
    return pred, margin


def bal_acc_abstain_wrong(y, pred):
    yM = np.asarray(y) == "M"
    rM = ((pred == "M") & yM).sum() / max(yM.sum(), 1)
    rB = ((pred == "B") & ~yM).sum() / max((~yM).sum(), 1)
    return (rM + rB) / 2


def min_extent_for(n, frac=0.02):
    return max(5, math.ceil(frac * n))


DELTAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
KS = [1, 3, 5, 10]
THETAS = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95]


def select_params(A, ya, relative=True, agg="topk", n_inner=5, seed=0, knots="consistent", min_frac=0.02, max_len=5):
    grid_d = DELTAS if relative else THETAS
    grid_k = KS if agg == "topk" else [1]
    sc = np.zeros((len(grid_d), len(grid_k)))
    fk = fit_knots_consistent if knots == "consistent" else fit_knots_percentile
    for tr, va in StratifiedKFold(n_inner, shuffle=True, random_state=seed).split(A, ya):
        e, kn = fit_tertile_edges(A[tr]), fk(A[tr])
        C = crisp_context(A[tr], e)
        cd = Candidates(C, ya[tr] == "M", min_extent_for(len(tr), min_frac), max_len)
        if len(cd.items) == 0:
            continue
        S = cd.soft(evidence(A[va], kn, "max"))
        for di, d in enumerate(grid_d):
            mask, isM, w = cd.select(d, relative)
            for ki, k in enumerate(grid_k):
                b, m, _, _ = aggregate(S, mask, isM, w, k, agg)
                pred, _ = decide(b, m)
                sc[di, ki] += bal_acc_abstain_wrong(ya[va], pred) / n_inner
    best = sc.max()
    ok = np.argwhere(sc >= best - 0.005)
    di, ki = max(ok.tolist(), key=lambda t: (t[0], -t[1]))  # parsimony: stricter filter, then smaller k
    return grid_d[di], grid_k[ki], sc


class SVN2:
    def __init__(self, relative=True, agg="topk", knots="consistent", closed_only=False, min_frac=0.02, max_len=5):
        self.relative, self.agg, self.knots, self.closed_only, self.min_frac, self.max_len = relative, agg, knots, closed_only, min_frac, max_len

    def fit(self, A, ya, delta=None, k=None, seed=0):
        if delta is None:
            delta, k, self.grid = select_params(A, ya, self.relative, self.agg, seed=seed, knots=self.knots, min_frac=self.min_frac, max_len=self.max_len)
        self.delta, self.k = delta, k
        self.edges = fit_tertile_edges(A)
        self.kn = (fit_knots_consistent if self.knots == "consistent" else fit_knots_percentile)(A)
        C = crisp_context(A, self.edges)
        self.cd = Candidates(C, ya == "M", min_extent_for(len(A), self.min_frac), self.max_len)
        mask, self.isM, self.w = self.cd.select(delta, self.relative)
        if self.closed_only:
            mask &= self.cd.closed
        self.mask = mask
        self.n_rules = int(mask.sum())
        self.n_rules_M = int((mask & self.isM).sum())
        return self

    def evidence_scores(self, B, mode="svn"):
        if len(self.cd.items) == 0:
            z = np.zeros(len(B)); return z, z, z, z
        if mode == "crisp":
            S = self.cd.crisp(crisp_context(B, self.edges))
        else:
            S = self.cd.soft(evidence(B, self.kn, {"svn": "max", "T": "none", "entropy": "entropy"}[mode]))
        return aggregate(S, self.mask, self.isM, self.w, self.k, self.agg)

    def predict(self, B, mode="svn"):
        b, m, ub, um = self.evidence_scores(B, mode)
        self.last_used = np.where(b > m, ub, um)
        pred, margin = decide(b, m)
        tot = b + m
        score = np.where(tot > 0, (m - b) / np.where(tot > 0, tot, 1), 0.0)
        return pred, margin, score


In [ ]:
%%writefile svn_eval.py
"""Evaluation helpers shared by both notebooks: baselines, one cross-validation fold, statistics."""
import warnings
import numpy as np, pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import wittgenstein as lw
from svnfca import fit_tertile_edges, fit_knots, crisp_context, mine_rules, RuleBase, run_svn, metrics
from svnfca2 import SVN2, bal_acc_abstain_wrong

warnings.filterwarnings("ignore")
MAIN = "SVN-FCA-IC"
_STORE = None


def top_k(A, ya, k=5):
    """Indices of the k features with the largest ANOVA F value (training fold only)."""
    if A.shape[1] <= k:
        return np.arange(A.shape[1])
    F, _ = f_classif(A, ya == "M")
    return np.sort(np.argsort(-np.nan_to_num(F))[:k])


def base_models(n_trees=300):
    sc = lambda m: make_pipeline(StandardScaler(), m)
    return {
        "LR": sc(LogisticRegression(max_iter=5000, random_state=42)),
        "DT (depth 5)": DecisionTreeClassifier(max_depth=5, random_state=42),
        "RF": RandomForestClassifier(n_estimators=n_trees, random_state=42),
        "SVM-RBF": sc(SVC(probability=True, random_state=42)),
        "kNN": sc(KNeighborsClassifier(5)),
        "GNB": GaussianNB(),
        "LR (balanced)": sc(LogisticRegression(max_iter=5000, random_state=42, class_weight="balanced")),
        "DT (depth 5, balanced)": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
        "RF (balanced)": RandomForestClassifier(n_estimators=n_trees, random_state=42, class_weight="balanced"),
        "SVM-RBF (balanced)": sc(SVC(probability=True, random_state=42, class_weight="balanced")),
    }


def ripper(A, ya, B):
    cn = [f"f{i}" for i in range(A.shape[1])]
    dfa = pd.DataFrame(A, columns=cn); dfa["y"] = (ya == "M")
    r = lw.RIPPER(random_state=0)
    r.fit(dfa, class_feat="y", pos_class=True)
    dfb = pd.DataFrame(B, columns=cn)
    pred = np.where(np.array(r.predict(dfb)), "M", "B").astype(object)
    return pred, np.asarray(r.predict_proba(dfb))[:, 1], len(r.ruleset_)


def _row(yb, pred, score, name, fi, rep, n_rules, feats, extra=None):
    d = metrics(yb, pred)
    d.update(model=name, fold=fi, rep=rep, n_rules=n_rules, feats=feats,
             auc=roc_auc_score(yb == "M", score), bal_all=bal_acc_abstain_wrong(yb, np.asarray(pred, dtype=object)))
    if extra:
        d.update(extra)
    if _STORE is not None:
        _STORE[(name, feats)] = (np.asarray(pred, dtype=object), np.asarray(score, float))
    return d


def one_fold(Xa, yv, tr, va, fi, rep, fixed_cols=None, ablations=False, refs_all=True, v1=True, n_trees=300):
    rows = []
    A_all, B_all, ya, yb = Xa[tr], Xa[va], yv[tr], yv[va]
    cols = np.asarray(fixed_cols) if fixed_cols is not None else top_k(A_all, ya)
    A, B = A_all[:, cols], B_all[:, cols]
    ft = str(len(cols))
    full = SVN2().fit(A, ya, seed=fi)
    p, mg, s = full.predict(B, "svn")
    rows.append(_row(yb, p, s, MAIN, fi, rep, full.n_rules, ft,
                     dict(delta=full.delta, kk=full.k, rules_used=float(np.mean(full.last_used)), n_rules_M=full.n_rules_M, margin_mean=float(np.mean(np.abs(mg))))))
    oof = pd.DataFrame(dict(fold=fi, rep=rep, y=yb, pred=p, margin=mg, score=s))
    cl = SVN2(closed_only=True).fit(A, ya, full.delta, full.k)
    p, _, s = cl.predict(B); rows.append(_row(yb, p, s, "Closed-intent rules", fi, rep, cl.n_rules, ft))
    if ablations:
        for name, mode in [("Crisp matching", "crisp"), ("Truth only (I = 0)", "T"), ("Entropy indeterminacy", "entropy")]:
            p, _, s = full.predict(B, mode); rows.append(_row(yb, p, s, name, fi, rep, full.n_rules, ft))
        for name, mdl in [("Absolute purity filter", SVN2(relative=False)), ("Sum of rule evidence", SVN2(agg="sum")), ("Percentile knots", SVN2(knots="percentile"))]:
            mdl.fit(A, ya, seed=fi); p, _, s = mdl.predict(B)
            rows.append(_row(yb, p, s, name, fi, rep, mdl.n_rules, ft, dict(delta=mdl.delta, kk=mdl.k)))
    if v1:  # original model: fixed purity 0.85, summed evidence, percentile knots
        e, k = fit_tertile_edges(A), fit_knots(A)
        C, Cb = crisp_context(A, e), crisp_context(B, e)
        rb = RuleBase(mine_rules(C, ya, 10, .85, 5), C.shape[1])
        p, mg1, b, m = run_svn(rb, B, Cb, k, "svn")
        tot = b + m
        rows.append(_row(yb, p, np.where(tot > 0, (m - b) / np.where(tot > 0, tot, 1), 0.0), "Fixed 0.85 purity (first version)", fi, rep, len(rb), ft))
    bm = base_models(n_trees)
    for name, mdl in bm.items():
        m_ = clone(mdl).fit(A, ya)
        rows.append(_row(yb, m_.predict(B), m_.predict_proba(B)[:, list(m_.classes_).index("M")], name, fi, rep, 0, ft))
    p, s, nr = ripper(A, ya, B); rows.append(_row(yb, p, s, "RIPPER", fi, rep, nr, ft))
    if refs_all and A_all.shape[1] > len(cols):
        for name in ["LR", "RF", "SVM-RBF", "LR (balanced)", "RF (balanced)", "SVM-RBF (balanced)"]:
            m_ = clone(bm[name]).fit(A_all, ya)
            rows.append(_row(yb, m_.predict(B_all), m_.predict_proba(B_all)[:, list(m_.classes_).index("M")], name, fi, rep, 0, "all"))
    return rows, oof


def run_cv(Xa, yv, n_repeats=10, seed=2024, **kw):
    rows, oofs = [], []
    for fi, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=seed).split(Xa, yv)):
        r, o = one_fold(Xa, yv, tr, va, fi, fi // 5, **kw)
        rows += r; oofs.append(o)
    return pd.DataFrame(rows), pd.concat(oofs, ignore_index=True)


def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty_like(p); run = 0.0
    for i, j in enumerate(o):
        run = max(run, (len(p) - i) * p[j]); out[j] = min(1.0, run)
    return out


def corrected_t(d, k_ratio=0.25):
    """Corrected resampled t-test (Nadeau & Bengio, 2003) on paired fold differences d."""
    d = np.asarray(d, float); k = len(d); v = d.var(ddof=1)
    if v <= 0:
        return 0.0, 1.0
    t = d.mean() / np.sqrt((1 / k + k_ratio) * v)
    return t, 2 * stats.t.sf(abs(t), k - 1)


def summarise(cv, ref=MAIN):
    cols = ["accuracy", "balanced_accuracy", "precision_M", "recall_M", "f1_M", "auc", "bal_all"]
    g = cv.groupby(["feats", "model"])
    out = g[cols + ["n_rules", "coverage"]].mean().join(g[["accuracy", "balanced_accuracy", "auc"]].std().add_suffix("_std")).reset_index()
    r = cv[cv.model == ref].sort_values("fold")
    for metric in ["accuracy", "balanced_accuracy", "auc"]:
        ps = []
        for (f, mname), gg in cv.groupby(["feats", "model"]):
            if mname == ref and f == r.feats.iloc[0]:
                ps.append(np.nan); continue
            d = r[metric].values - gg.sort_values("fold")[metric].values
            ps.append(corrected_t(d)[1])
        out["p_" + metric] = ps
        mk = out["p_" + metric].notna().values
        out.loc[mk, "p_" + metric + "_holm"] = holm(out.loc[mk, "p_" + metric].values)
    return out


## Datasets

In [ ]:
import svn_eval
from svn_eval import *
from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from sklearn.datasets import load_breast_cancer, fetch_openml
GH = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
PM = "https://media.githubusercontent.com/media/EpistasisLab/pmlb/master/datasets/"
# name -> (role, loader). M = positive (disease or minority) class, B = the other class.
def wdbc():
    d = load_breast_cancer(); return d.data[:, :5], np.where(d.target == 0, "M", "B"), {}
def pima():
    d = pd.read_csv(GH + "pima-indians-diabetes.data.csv", header=None); return d.iloc[:, :8].values.astype(float), np.where(d[8] == 1, "M", "B"), {}
def haberman():
    d = pd.read_csv(GH + "haberman.csv", header=None); return d.iloc[:, :3].values.astype(float), np.where(d[3] == 2, "M", "B"), {"positive": "died within five years"}
def pmlb(name):
    def f():
        d = pd.read_csv(f"{PM}{name}/{name}.tsv.gz", sep="\t"); return d.drop(columns="target").values.astype(float), np.where(d.target == 1, "M", "B"), {}
    return f
def ionosphere():
    d = pd.read_csv(GH + "ionosphere.csv", header=None); return d.iloc[:, :34].drop(columns=[1]).values.astype(float), np.where(d[34] == "b", "M", "B"), {"note": "constant column 1 removed"}
def openml(name, positive):
    def f():
        d = fetch_openml(name, version=1, as_frame=True, parser="auto"); X, t = d.data, d.target.astype(str)
        num = X.select_dtypes(include="number"); keep = ~num.isna().any(axis=1); info = dict(dropped_non_numeric=[c for c in X.columns if c not in num.columns], classes=t.value_counts().to_dict())
        pos = positive if positive in set(t) else t.value_counts().idxmin(); info["positive_label"] = pos
        return num[keep].values.astype(float), np.where(t[keep].values == pos, "M", "B"), info
    return f
DATASETS = {
    "WDBC (first five mean features)": ("reference", wdbc),
    "Pima diabetes": ("development", pima), "Haberman survival": ("development", haberman), "Bupa liver": ("development", pmlb("bupa")),
    "Appendicitis": ("development", pmlb("appendicitis")), "Ionosphere": ("development", ionosphere),
    "Statlog heart": ("confirmation", openml("heart-statlog", "present")), "Blood transfusion": ("confirmation", openml("blood-transfusion-service-center", "2")),
    "Indian liver (ILPD)": ("confirmation", openml("ilpd", "1")),
}

## Cross-validation on every dataset

In [ ]:
t0 = time.time(); allcv, allsum, meta = [], [], {}
for name, (role, loader) in DATASETS.items():
    try:
        Xa, yv, info = loader()
        ok_ = ~np.isnan(Xa).any(axis=1); Xa, yv = Xa[ok_], yv[ok_]
    except Exception as e:
        print(f"!! {name} could not be loaded: {type(e).__name__}: {e}"); continue
    print(f"\n=== {name} [{role}]: {Xa.shape[0]} cases, {Xa.shape[1]} numeric features, positive share {np.mean(yv == 'M'):.3f} {info}")
    cvd, _ = run_cv(Xa, yv, n_repeats=10, seed=2024, fixed_cols=(list(range(5)) if name.startswith("WDBC") else None), ablations=False, refs_all=True, v1=True)
    cvd["dataset"] = name; s = summarise(cvd); s.insert(0, "dataset", name); s.insert(1, "role", role)
    allcv.append(cvd); allsum.append(s); meta[name] = dict(role=role, n=int(Xa.shape[0]), p=int(Xa.shape[1]), positive_share=float(np.mean(yv == "M")), **info)
    print(s[["feats", "model", "accuracy", "balanced_accuracy", "auc", "n_rules", "coverage", "p_balanced_accuracy_holm"]].round(3).to_string(index=False)); print("elapsed", round(time.time() - t0), "s")
cvall = pd.concat(allcv, ignore_index=True); summ = pd.concat(allsum, ignore_index=True)
cvall.to_csv("results/cv_folds_all_datasets.csv", index=False); summ.to_csv("results/summary_all_datasets.csv", index=False); json.dump(meta, open("results/dataset_info.json", "w"), indent=1, default=str)

## Tests across datasets (Friedman, Wilcoxon with Holm correction)

In [ ]:
def rank_table(metric):
    five = summ[summ.feats != "all"].pivot_table(index="dataset", columns="model", values=metric)
    five = five.dropna(axis=1)
    ranks = five.apply(lambda r: pd.Series(rankdata(-r.values), index=r.index), axis=1)
    return five, ranks
out = {}
for metric in ["balanced_accuracy", "accuracy", "auc"]:
    five, ranks = rank_table(metric)
    stat, p = friedmanchisquare(*[five[c].values for c in five.columns])
    avg = ranks.mean().sort_values(); pw = {}
    for c in five.columns:
        if c == MAIN: continue
        try: pw[c] = float(wilcoxon(five[MAIN].values, five[c].values).pvalue)
        except ValueError: pw[c] = 1.0
    hp = dict(zip(pw.keys(), holm(list(pw.values()))))
    print(f"\n--- {metric}: {five.shape[0]} datasets, {five.shape[1]} models; Friedman chi2 = {stat:.3f}, p = {p:.4g}")
    print(pd.DataFrame(dict(avg_rank=avg.round(2), wilcoxon_p_vs_main=pd.Series(pw), holm_p=pd.Series(hp))).loc[avg.index].round(4).to_string())
    five.round(4).to_csv(f"results/mean_{metric}_by_dataset.csv"); out[metric] = dict(friedman_chi2=float(stat), friedman_p=float(p), n_datasets=int(five.shape[0]), avg_rank=avg.to_dict(), wilcoxon_p=pw, holm_p=hp)
json.dump(out, open("results/cross_dataset_tests.json", "w"), indent=1)
print("\nMain-model coverage and rules per dataset:"); print(summ[(summ.model == MAIN)][["dataset", "role", "accuracy", "balanced_accuracy", "auc", "coverage", "n_rules"]].round(3).to_string(index=False))

## Download

In [ ]:
import shutil
shutil.make_archive("svnfca_v2_extra_datasets_results", "zip", "results"); print("Created svnfca_v2_extra_datasets_results.zip with:", sorted(os.listdir("results")))
try:
    from google.colab import files
    files.download("svnfca_v2_extra_datasets_results.zip")
except Exception as e:
    print("Automatic download not available:", e, "\nUse the folder icon on the left, open the zip file, and download it.")